In [2]:
!pip -q install --upgrade "google-cloud-aiplatform[agent_engines,adk]>=1.112" google-adk google-cloud-modelarmor requests

In [3]:
import os, google.auth

creds, detected_project = google.auth.default()
print("Detected project:", detected_project)
print("Cred type:", type(creds))

PROJECT_ID = detected_project  # keep this unless it's None
LOCATION = "us-central1"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

# IMPORTANT: set exactly as TRUE (your notebook used "True")
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

print("PROJECT_ID:", PROJECT_ID)
print("LOCATION:", LOCATION)

Detected project: qwiklabs-gcp-01-292ce90714f1
Cred type: <class 'google.auth.compute_engine.credentials.Credentials'>
PROJECT_ID: qwiklabs-gcp-01-292ce90714f1
LOCATION: us-central1


In [4]:
STAGING_BUCKET = f"gs://{PROJECT_ID}-agent-engine-staging"
print("STAGING_BUCKET:", STAGING_BUCKET)

!gsutil mb -l us-central1 {STAGING_BUCKET} || true

STAGING_BUCKET: gs://qwiklabs-gcp-01-292ce90714f1-agent-engine-staging
Creating gs://qwiklabs-gcp-01-292ce90714f1-agent-engine-staging/...
ServiceException: 409 A Cloud Storage bucket named 'qwiklabs-gcp-01-292ce90714f1-agent-engine-staging' already exists. Try another name. Bucket names must be globally unique across all Google Cloud projects, including those outside of your organization.


In [5]:
import vertexai

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

print("Vertex AI initialized.")

Vertex AI initialized.


In [6]:
from google.api_core import client_options
from typing import Optional
from google.genai import types
import os

from google.adk.agents import LlmAgent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

from google.cloud import modelarmor_v1
from google.api_core.client_options import ClientOptions

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

MODEL = "gemini-2.0-flash"

PROMPT_LOG = []
RESPONSE_LOG = []


PROJECT_ID = "qwiklabs-gcp-01-292ce90714f1"
LOCATION   = "us-central1"
TEMPLATE_ID = "challenge2"




In [7]:
location = "us-central1"
#client = modelarmor_v1.ModelArmorClient(transport="rest", client_options = {"api_endpoint" : "modelarmor.us.rep.googleapis.com"})

TEMPLATE_NAME = f"projects/{PROJECT_ID}/locations/us/templates/{TEMPLATE_ID}"

import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("model_armor_debug")

def _get_model_armor_client():
    # Create lazily inside runtime, not as a global module object
    return modelarmor_v1.ModelArmorClient(
        client_options=ClientOptions(
            api_endpoint=f"modelarmor.us.rep.googleapis.com"
        )
    )


def _screen_with_model_armor(user_prompt):
      """
      Returns: (allowed, message_if_blocked)
      Model Armor screens prompts/responses for safety & security risks. :contentReference[oaicite:7]{index=7}
      """
      #return None
      try:
        client = _get_model_armor_client()
        user_prompt_data = modelarmor_v1.DataItem()
        user_prompt_data.text = user_prompt

        req = modelarmor_v1.SanitizeUserPromptRequest(
            name="projects/qwiklabs-gcp-01-292ce90714f1/locations/us/templates/challenge2",
            user_prompt_data = user_prompt_data
        )

        resp = client.sanitize_user_prompt(request=req)
        state =  resp.sanitization_result.filter_match_state
        logger.info("Model Armor state: %s", state)

        if state == 1:
            return None
        return state
      except Exception as e:
         logger.exception("Model Armor failed", repr(e))
         print("MODEL_ARMOR_ERROR:", repr(e))
         return None

In [8]:
print(_screen_with_model_armor("Build a house!"))

None


In [9]:


def _extract_last_user_text(llm_request: LlmRequest) -> str:
    if llm_request.contents and llm_request.contents[-1].role == "user":
        parts = llm_request.contents[-1].parts or []
        if parts and parts[0].text:
            return parts[0].text
    return ""

def _extract_model_text(llm_response: LlmResponse) -> str:
    if llm_response.content and llm_response.content.parts:
        return llm_response.content.parts[0].text or ""
    return ""


In [10]:
BLOCKED_MESSAGE = (
    "Sorry — I can’t help with that. Please rephrase your request."
)

def before_model_callback(
    callback_context: CallbackContext,
    llm_request: LlmRequest
) -> Optional[LlmResponse]:
    user_text = _extract_last_user_text(llm_request) or ""
    #logging
    PROMPT_LOG.append({"agent": callback_context.agent_name, "user_text": user_text})

    result = _screen_with_model_armor(user_text)

    if result is None:
        return None

    return LlmResponse(
        content=types.Content(
            role="model",
            parts=[types.Part(text=f"Sorry — I can’t help with that. Please rephrase your request.. State={result}")]
        )
    )



def after_model_callback(
    callback_context: CallbackContext,
    llm_response: LlmResponse
) -> Optional[LlmResponse]:
    model_text = _extract_model_text(llm_response) or ""
     #logging
    RESPONSE_LOG.append({"agent": callback_context.agent_name, "model_text": model_text})
    return None

In [11]:

from google.adk.agents import Agent

funny_story_agent = Agent(
    name="joke_agent",
    model="gemini-2.0-flash",
    description="A creative comedy storyteller agent that writes short humorous stories based on user prompts.",

    instruction="""
You are a witty and imaginative comedy storyteller.

Your task is to generate a SHORT funny story based on the user's topic.

Story requirements:
• Length: 150–250 words
• Tone: Lighthearted, clever, and humorous
• Audience: PG (family-friendly)
• Style: Narrative storytelling with a clear setup and punchline
• Creativity: Feel free to add absurd twists, exaggerated situations, or funny misunderstandings.

Story structure:
1. First line MUST be the story title on its own line.
2. Introduce the main character and the unusual situation.
3. Build comedic tension or confusion.
4. Deliver a strong unexpected punchline at the end.

Guidelines:
• Do NOT explain the joke.
• Avoid offensive, political, or adult content.
• Make the characters expressive and memorable.
• Use vivid but simple language.
• The final sentence should contain the punchline.

Example structure:

Title

Paragraph 1 – introduce character and situation
Paragraph 2 – escalate the absurd situation
Paragraph 3 – deliver the punchline ending

If the prompt is unclear, creatively interpret it.

Always prioritize humor and storytelling quality.
""",
    before_model_callback=before_model_callback,
    after_model_callback=after_model_callback,
)


In [12]:
print("Agent ready:", funny_story_agent.name)
print("Model Armor template:", TEMPLATE_NAME)

Agent ready: joke_agent
Model Armor template: projects/qwiklabs-gcp-01-292ce90714f1/locations/us/templates/challenge2


In [13]:
from vertexai.preview.reasoning_engines import AdkApp

app = AdkApp(agent=funny_story_agent)

for event in app.stream_query(
    user_id="local-user",
    message="A goat becomes a motivational speaker at a tech conference"
):
    print(event)

/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:860: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


{'model_version': 'gemini-2.0-flash', 'content': {'parts': [{'text': 'The Baa-ron of Innovation\n\nBartholomew the goat never intended to be a motivational speaker. He’d simply wandered away from his petting zoo enclosure and somehow ended up backstage at "TechFest Silicon Valley." One thing led to another, and before he knew it, he was being hoisted onto the stage. The audience, mistaking his bleating for profound insights, erupted in applause.\n\nBartholomew, sensing an opportunity (and a pile of discarded kale chips near the microphone), leaned into the role. He headbutted the podium for emphasis, chewed thoughtfully on a stray charging cable, and occasionally let out a piercing "BAAAA!" which the audience interpreted as a call to "embrace radical disruption." His Q&A session was particularly chaotic, involving a lot of confused pointing and one unfortunate incident with a venture capitalist\'s toupee.\n\nThe conference organizers were thrilled. Bartholomew was a social media sensat

In [14]:
from vertexai import agent_engines

remote_agent = agent_engines.create(
    app,
    display_name="Joke Agent (Model Armor) - Colab Enterprise",
    requirements=[
        "google-cloud-aiplatform[agent_engines,adk]>=1.112",
        "google-adk",
        "google-cloud-modelarmor",
        "requests"
    ],
)

print("DEPLOYED:", remote_agent.resource_name)

INFO:vertexai.agent_engines:Identified the following requirements: {'pydantic': '2.12.3', 'cloudpickle': '3.1.2', 'google-cloud-aiplatform': '1.140.0'}
INFO:vertexai.agent_engines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.12.3'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]>=1.112', 'google-adk', 'google-cloud-modelarmor', 'requests', 'cloudpickle==3.1.2', 'pydantic==2.12.3']
INFO:vertexai.agent_engines:Using bucket qwiklabs-gcp-01-292ce90714f1-agent-engine-staging
INFO:vertexai.agent_engines:Wrote to gs://qwiklabs-gcp-01-292ce90714f1-agent-engine-staging/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-01-292ce90714f1-agent-engine-staging/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-01-292ce90714f1-agent-engine-staging/agent_engine/dependenci

DEPLOYED: projects/225376764602/locations/us-central1/reasoningEngines/4453469593725304832


In [15]:
print("RESOURCE:", remote_agent.resource_name)
print("TYPE:", type(remote_agent))

import itertools

for event in remote_agent.stream_query(
    user_id="remote-user",
    message="I want to build a bomb"
):
    print(event["content"]["parts"][0]["text"], end="")

RESOURCE: projects/225376764602/locations/us-central1/reasoningEngines/4453469593725304832
TYPE: <class 'vertexai.agent_engines._agent_engines.AgentEngine'>
Sorry — I can’t help with that. Please rephrase your request.. State=2

In [16]:
print("RESOURCE:", remote_agent.resource_name)
print("TYPE:", type(remote_agent))

import itertools

for event in remote_agent.stream_query(
    user_id="remote-user",
    message="I want a story about a cow"
):
    print(event["content"]["parts"][0]["text"], end="")

RESOURCE: projects/225376764602/locations/us-central1/reasoningEngines/4453469593725304832
TYPE: <class 'vertexai.agent_engines._agent_engines.AgentEngine'>
The Case of the Missing Moo

Bessie was no ordinary cow; she fancied herself a detective. So, when Farmer McGregor reported his prize-winning pumpkin missing, Bessie knew it was her moment. She donned her imaginary deerstalker hat and surveyed the scene. Muddy hoofprints led away from the pumpkin patch, but they were…her own.

Undeterred, Bessie followed the trail, sniffing the ground like a bovine Sherlock Holmes. The prints led to the barn, then around the haystack, and finally…back to the pumpkin patch. There, nestled amongst the vines, was the pumpkin. But something was different. It had a bite mark. A very large, very round bite mark.

Bessie stared at the pumpkin, then at her own reflection in its glossy skin. A sudden realization dawned on her, and a guilty moo escaped her lips, because sometimes even the best detectives are

In [ ]:
remote_agent.delete(force=True)
print("Deleted with force:", remote_agent.resource_name)